# Week 1 Exercise — Personal AI Technical Tutor

**Author:** slavapa

A reusable tutor for this course: type a technical question (or paste confusing code),
get a clear explanation from **OpenAI** and/or **Ollama**.

Goes beyond the starter solution with:
- A system prompt tuned for an LLM-engineering student
- Multi-shot examples (good + bad explanation styles)
- Streaming for OpenAI; same OpenAI-compatible client for local Ollama
- Easy model swap (`llama3.2`, `qwen2.5`, etc.)

**Setup**
1. `.env` with `OPENAI_API_KEY` (for GPT)
2. Ollama running at [http://localhost:11434/](http://localhost:11434/) and `ollama pull llama3.2`

In [ ]:
# imports

import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"  # try "qwen2.5", "deepseek-r1", "llama3.2:1b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# set up environment

load_dotenv(override=True)
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Ollama:", requests.get("http://localhost:11434").content)

## Question

Edit `question` below, or use the `input()` cell afterward for interactive use.

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# optional: interactive question (uncomment to use)
# question = input("Ask your technical tutor: ")

## Prompts (personalized + multi-shot)

The system prompt encodes *how* I want explanations: assume Python + LLM-course context,
prefer short concrete examples, and avoid jargon without definition.
Few-shot messages steer style — good answer vs. what to avoid.

In [ ]:
system_prompt = """
You are my personal technical tutor for the LLM Engineering course.

Audience:
- Comfortable with Python fundamentals (loops, dicts, generators)
- Learning LLMs: tokens, prompts, chat completions, streaming, Ollama
- Prefers clear, structured explanations over hand-wavy analogies

How to answer:
1. Start with a one-sentence plain-English summary.
2. Walk through the idea (or code) step by step.
3. Call out pitfalls or common misconceptions.
4. End with a tiny related example or a "try this next" tip when useful.
5. Respond in markdown. Do not wrap the whole reply in a markdown code fence.

If the question includes code, explain what it does AND why that pattern exists.
""".strip()

# Multi-shot: show the style we want (and a counter-example of what not to do)
few_shot = [
    {
        "role": "user",
        "content": "What does `stream=True` do in openai.chat.completions.create?",
    },
    {
        "role": "assistant",
        "content": """**Summary:** `stream=True` returns the reply in chunks as the model generates them, instead of waiting for the full answer.

**Why it matters:** In a notebook you can update a Markdown display live — better UX and you see progress on long answers.

**How to use it:** Iterate the stream and append each `chunk.choices[0].delta.content` (skip `None`).

**Pitfall:** Streaming responses don't give you one finished string; you assemble it yourself.

**Try next:** Compare the same prompt with and without streaming and watch latency-to-first-token.""",
    },
    {
        "role": "user",
        "content": "Explain tokens poorly so I can tell you what NOT to do.",
    },
    {
        "role": "assistant",
        "content": (
            "(Counter-example — do NOT answer like this:) "
            "Tokens are like magic AI words and the model just knows stuff. Trust the vibes."
        ),
    },
    {
        "role": "user",
        "content": "Now explain tokens the right way.",
    },
    {
        "role": "assistant",
        "content": """**Summary:** A token is a chunk of text the model reads/writes — often a word piece, not always a full word.

**Why it matters:** API cost and context limits are measured in tokens. Rough English rule of thumb: ~4 characters ≈ 1 token.

**Pitfall:** Code, punctuation, and non-English text often use more tokens than you expect.

**Try next:** Paste a prompt into a tokenizer visualizer and count tokens before calling the API.""",
    },
]


def build_messages(q: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        *few_shot,
        {
            "role": "user",
            "content": f"Please give a detailed explanation of the following:\n\n{q.strip()}",
        },
    ]


messages = build_messages(question)
print(f"Message count (system + few-shot + question): {len(messages)}")

## OpenAI (`gpt-4o-mini`) — streaming

In [ ]:
def ask_openai(q: str, model: str = MODEL_GPT) -> str:
    """Stream an explanation from OpenAI and display as Markdown."""
    stream = openai.chat.completions.create(
        model=model,
        messages=build_messages(q),
        stream=True,
    )
    response = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=handle.display_id)
    return response


print(f"### {MODEL_GPT}\\n")
gpt_reply = ask_openai(question)

## Ollama (`llama3.2`) — local, OpenAI-compatible API

Same client pattern as Day 2 — no cloud cost; swap `MODEL_LLAMA` to compare models.

In [ ]:
def ask_ollama(q: str, model: str = MODEL_LLAMA) -> str:
    """Get an explanation from a local Ollama model and display as Markdown."""
    result = ollama.chat.completions.create(
        model=model,
        messages=build_messages(q),
    )
    reply = result.choices[0].message.content
    display(Markdown(f"**Model:** `{model}`\n\n{reply}"))
    return reply


print(f"### {MODEL_LLAMA}\\n")
llama_reply = ask_ollama(question)

## Optional: compare another local model

Pull first if needed, e.g. `ollama pull qwen2.5` or `ollama pull deepseek-r1`.

In [ ]:
# Uncomment to try another local model:
# ask_ollama(question, model="qwen2.5")

## How to keep improving this tutor

1. When an answer is excellent, paste a shortened version into `few_shot`.
2. When an answer is bad, add a short counter-example ("don't explain like this") then the fixed version.
3. Tune `system_prompt` for your background (e.g. more/less math, more Kabbalah-domain metaphors, etc.).
4. After Week 2, wrap `ask_openai` / `ask_ollama` in a Gradio UI.